# Purpose:
- Curate all the datasets collected in lims for GCaMP8 characterization
    - GCaMP7 data for comparison
    - Only for STAGE_1 (omFISH project) and OPHYS_1_passive_images_A sessions
    - Only from VISp
- Document error data, and why
- Compare with what's in the codeocean
- Follow up from 240701_query_lims_data.ipynb

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
from brain_observatory_qc.data_access import from_lims

from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache as bpc

from pymongo import MongoClient
mongo = MongoClient("flaskapp.corp.alleninstitute.org", 27017)

cache = bpc.from_lims()
table = cache.get_ophys_experiment_table(passed_only=False)

In [ ]:
mids_gad2_ai195 = [622537, 651007, 633542]
mids_gad2_ai210 = [628165, 622756, 612771]

group_mid_dict = {'gad2_slc32a1_ai195': mids_gad2_ai195,
                  'gad2_slc32a1_ai210': mids_gad2_ai210,

}

remove_session_ids = [1303235340, 1299462513]  # 1303235340 was a test session. 1299462513 had very low signal (don't know why)
remove_acquisition_dates = ['2024-08-14', '2024-09-24']  # weird power setting

gcamp_table_pre = pd.DataFrame()
problems_table = []
for key, val in group_mid_dict.items():
    temp_table, temp_problems = get_session_info_per_group_of_mice(val, key)
    gcamp_table_pre = pd.concat([gcamp_table_pre, temp_table])
    problems_table.extend(temp_problems)